# NF-v3 temporal graph construction

This notebook is intentionally thin. The tested graph builder lives in `code/python/scripts/build_nfv3_graphs.py`; this notebook only mounts Drive, configures paths, runs a small smoke build, and inspects its audit.

Run the smoke build first. Do not run the full build until its generated schema, mappings, provenance, and audit have been reviewed.

In [29]:

# ============================================================
# SETUP - Run this cell first
# ============================================================
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path('/content/temporalgnn-nids')
REPO_URL = 'https://github.com/tatipar/temporalgnn-nids.git'
BRANCH = 'feat/fair-retrain-clean'

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)], check=True)

sys.path.append(str(REPO_ROOT / 'code/python'))

In [30]:
!pip install -q torch-geometric

from google.colab import drive
drive.mount('/content/drive')

import json

PROJECT_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain')
CORRECTED_ROOT = PROJECT_ROOT / 'corrected_data' / 'infiltration_v1'
CORRECTED_CSV = CORRECTED_ROOT / 'nfv3_corrected.csv'
CORRECTED_MANIFEST = CORRECTED_ROOT / 'nfv3_corrected.manifest.json'

assert REPO_ROOT.is_dir(), f'Repository not found: {REPO_ROOT}'
assert CORRECTED_CSV.is_file(), f'Corrected CSV not found: {CORRECTED_CSV}'
assert CORRECTED_MANIFEST.is_file(), f'Corrected manifest not found: {CORRECTED_MANIFEST}'

GRAPH_VERSION = 'infiltration_v1_w30_tcpflags_v1'
PREFLIGHT_ROOT = PROJECT_ROOT / 'graphs' / f'{GRAPH_VERSION}_preflight'
SMOKE_ROOT = PROJECT_ROOT / 'graphs' / f'{GRAPH_VERSION}_smoke'
FULL_ROOT = PROJECT_ROOT / 'graphs' / GRAPH_VERSION
PROFILES = ('nfv3_extended', 'portable_core')

print(f'Corrected CSV: {CORRECTED_CSV}')
print(f'Preflight output: {PREFLIGHT_ROOT}')
print(f'Smoke output: {SMOKE_ROOT}')
print(f'Full output: {FULL_ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Corrected CSV: /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv
Preflight output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_preflight
Smoke output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_smoke
Full output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1


In [31]:
%cd /content/temporalgnn-nids/code/python
!python -m unittest discover -s tests -p 'test_*.py' -v
%cd /content/temporalgnn-nids

/content/temporalgnn-nids/code/python
test_empty_windows_are_not_emitted (test_build_nfv3_graphs.CompleteWindowIteratorTests.test_empty_windows_are_not_emitted) ... ok
test_rejects_flow_start_regression_between_chunks (test_build_nfv3_graphs.CompleteWindowIteratorTests.test_rejects_flow_start_regression_between_chunks) ... ok
test_varied_durations_remain_ordered_across_chunk_boundaries (test_build_nfv3_graphs.CompleteWindowIteratorTests.test_varied_durations_remain_ordered_across_chunk_boundaries) ... ok
test_aligned_profiles_read_provenance_only_once (test_build_nfv3_graphs.OutputAuditTests.test_aligned_profiles_read_provenance_only_once) ... ok
test_collection_digest_is_order_independent_and_path_sensitive (test_build_nfv3_graphs.OutputAuditTests.test_collection_digest_is_order_independent_and_path_sensitive) ... ok
test_checkpoint_publishes_mapping_before_state (test_build_nfv3_graphs.ResumeMappingTests.test_checkpoint_publishes_mapping_before_state) ... ok
test_replay_uses_sources_

In [ ]:
preflight_command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(PREFLIGHT_ROOT),
    '--chunksize', '250000',
    '--preflight-only',
    '--overwrite',
]
for profile in PROFILES:
    preflight_command += ['--profile', profile]

print(' '.join(preflight_command))
subprocess.run(preflight_command, check=True)

preflight = json.loads((PREFLIGHT_ROOT / 'feature_preflight.json').read_text())
print('Preflight status:', preflight['status'])
print('Input rows / positives:', preflight['input_rows'], preflight['positive_rows'])
print('Retained rows / positives:', preflight['retained_rows'], preflight['retained_positive_rows'])
print('Excluded rows / positives:', preflight['excluded_rows'], preflight['excluded_positive_rows'])
print('Counts by source file:', json.dumps(preflight['by_source_file'], indent=2))
print('Invalid endpoint rows (union):', preflight['invalid_any_endpoint_rows'])
print('Invalid endpoint positive rows:', preflight['invalid_any_endpoint_positive_rows'])
print('Invalid source endpoint rows:', preflight['invalid_source_endpoint_rows'])
print('Invalid destination endpoint rows:', preflight['invalid_destination_endpoint_rows'])
print('Invalid source endpoint reasons:', preflight['invalid_source_endpoint_reasons'])
print('Invalid destination endpoint reasons:', preflight['invalid_destination_endpoint_reasons'])
print('Invalid ports:', preflight['invalid_port_rows'])
print('Invalid protocols:', preflight['invalid_protocol_rows'])
print('Invalid TCP flag bitmasks:', preflight['invalid_tcp_flags_rows'])
print('Non-TCP rows with non-zero TCP flags:', preflight['non_tcp_nonzero_tcp_flags_rows'])
print('Invalid binary targets:', preflight['invalid_binary_target_rows'])
print('Invalid time/duration rows:', preflight['invalid_time_or_duration_rows'])
print('Flow-end differences >1 ms:', preflight['flow_end_difference_gt_1ms_rows'])
print('Maximum absolute flow-end difference (ms):', preflight['flow_end_max_absolute_difference_ms'])
print('Invalid numeric rows:', preflight['invalid_numeric_rows_by_profile'])
print('Port categories:', preflight['port_category_counts'])
print('Port zero by protocol:', preflight['port_zero_by_protocol'])
print('Top other privileged ports:', preflight['other_privileged_top_ports'])
print('Top other high ports:', preflight['other_high_top_ports'])

In [32]:
command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(SMOKE_ROOT),
    '--chunksize', '250000',
    '--max-windows', '10',
    '--overwrite',
]
for profile in PROFILES:
    command += ['--profile', profile]

print(' '.join(command))
subprocess.run(command, check=True)

python /content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py --input-csv /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv --corrected-manifest /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json --output-root /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_smoke --chunksize 250000 --max-windows 10 --overwrite --profile nfv3_extended --profile portable_core


CompletedProcess(args=['python', '/content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py', '--input-csv', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv', '--corrected-manifest', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json', '--output-root', '/content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1_smoke', '--chunksize', '250000', '--max-windows', '10', '--overwrite', '--profile', 'nfv3_extended', '--profile', 'portable_core'], returncode=0)

In [33]:
audit = json.loads((SMOKE_ROOT / 'graph_audit.json').read_text())
configuration = json.loads((SMOKE_ROOT / 'build_configuration.json').read_text())

print('Audit status:', audit['status'])
print('Day-1 cutoffs:', configuration['day1_cutoffs'])
print('Day-1 decision-time distribution:')
print(json.dumps(configuration['day1_cutoffs']['decision_time_distribution'], indent=2))
print('Profile schema hashes:')
for name, digest in configuration['profiles'].items():
    print(f'- {name}: {digest}')

for day_name in ('day1', 'day2'):
    mapping = json.loads((SMOKE_ROOT / 'mappings' / f'{day_name}_ip_to_id.json').read_text())
    print(f"{day_name}: {mapping['entries']} mapped IPs; policy={mapping['creation_policy']}")

print('Artifact checksum summary:')
print(json.dumps(audit['artifacts'], indent=2))

Audit status: partial
Day-1 cutoffs: {'decision_time_distribution': {'full_span_ms': 85710000, 'lower_tail_span_ms': 43710000, 'maximum_ms': 1519862490000, 'minimum_ms': 1519776780000, 'p0_1_ms': 1519820490000, 'p99_9_ms': 1519858380000, 'quantile_method': 'nearest_rank', 'rows': 2022007, 'upper_tail_span_ms': 4110000}, 'raw_train_end_ms': 1519836776999, 'raw_val_end_ms': 1519849633500, 'train_end_ms': 1519836780000, 'val_end_ms': 1519849650000}
Day-1 decision-time distribution:
{
  "full_span_ms": 85710000,
  "lower_tail_span_ms": 43710000,
  "maximum_ms": 1519862490000,
  "minimum_ms": 1519776780000,
  "p0_1_ms": 1519820490000,
  "p99_9_ms": 1519858380000,
  "quantile_method": "nearest_rank",
  "rows": 2022007,
  "upper_tail_span_ms": 4110000
}
Profile schema hashes:
- nfv3_extended: 30e214b162eecc4c2578c658a5711534d22913d2279cb032429cc38df6c224c3
- portable_core: 95bf10470b10ba9039ce0753881f4743809dd583b8c04e8b7dc1c72706b76ae5
day1: 27 mapped IPs; policy=append_only_first_valid_chro

In [34]:
import pandas as pd
import torch

sample_provenance = next((SMOKE_ROOT / 'provenance' / 'day1').glob('graph_*.csv'))
sample_graph = SMOKE_ROOT / 'nfv3_extended' / 'train' / f'{sample_provenance.stem}.pt'
provenance = pd.read_csv(sample_provenance)
graph = torch.load(sample_graph, weights_only=False)

print('Graph:', sample_graph.name)
print('Nodes:', graph.num_nodes)
print('Edges:', graph.edge_index.shape[1])
print('Edge feature shape:', tuple(graph.edge_attr.shape))
print('Decision time (UTC epoch ms):', graph.timestamp)
display(provenance.head())

Graph: graph_1519776780000.pt
Nodes: 2
Edges: 1
Edge feature shape: (1, 40)
Decision time (UTC epoch ms): 1519776780000


,flow_id,source_file,source_row_id,source_ip,destination_ip,source_global_id,destination_global_id,edge_position,flow_start_ms,flow_end_ms,decision_time_ms,window_start_ms,window_end_ms,window_wait_ms,split,binary_target
0,cicids2018v3_wed2802.csv:0,cicids2018v3_wed2802.csv,0,5.188.9.25,172.31.64.89,0,1,0,1.519777e+12,1.519777e+12,1519776780000,1519776750000,1519776780000,4146.0,train,0


In [ ]:
from pathlib import Path
import json
import shutil

### 1. Prepare the simulation

SOURCE_SMOKE = (
    PROJECT_ROOT / "graphs"
    / "infiltration_v1_w30_tcpflags_v1_smoke"
)

RESUME_TEST_ROOT = Path("/content/nfv3_resume_test")
REFERENCE_ROOT = Path("/content/nfv3_reference_15")

for path in (RESUME_TEST_ROOT, REFERENCE_ROOT):
    if path.exists():
        shutil.rmtree(path)

shutil.copytree(SOURCE_SMOKE, RESUME_TEST_ROOT)

state_path = RESUME_TEST_ROOT / "build_state.json"
state = json.loads(state_path.read_text())

day1_graphs = sorted(
    (RESUME_TEST_ROOT / "nfv3_extended" / "train").glob("graph_*.pt")
)
day2_graphs = sorted(
    (RESUME_TEST_ROOT / "nfv3_extended" / "test2").glob("graph_*.pt")
)

assert len(day1_graphs) == 10
assert len(day2_graphs) == 10

day1_fifth = int(day1_graphs[4].stem.split("_")[1])
day2_fifth = int(day2_graphs[4].stem.split("_")[1])

state["days"]["day1"]["last_completed_decision_time_ms"] = day1_fifth
state["days"]["day2"]["last_completed_decision_time_ms"] = day2_fifth
state["completed"] = False

state_path.write_text(json.dumps(state, indent=2, sort_keys=True) + "\n")

print("Simulated checkpoint:")
print("day1:", day1_fifth)
print("day2:", day2_fifth)

### 2. Resume from window 5 to 15

resume_command = [
    "python",
    str(REPO_ROOT / "code/python/scripts/build_nfv3_graphs.py"),
    "--input-csv", str(CORRECTED_CSV),
    "--corrected-manifest", str(CORRECTED_MANIFEST),
    "--output-root", str(RESUME_TEST_ROOT),
    "--chunksize", "250000",
    "--max-windows", "10",
    "--resume",
]

for profile in PROFILES:
    resume_command += ["--profile", profile]

print(" ".join(resume_command))
subprocess.run(resume_command, check=True)

### 3. Build the continuous reference of 15 windows

reference_command = [
    "python",
    str(REPO_ROOT / "code/python/scripts/build_nfv3_graphs.py"),
    "--input-csv", str(CORRECTED_CSV),
    "--corrected-manifest", str(CORRECTED_MANIFEST),
    "--output-root", str(REFERENCE_ROOT),
    "--chunksize", "250000",
    "--max-windows", "15",
    "--overwrite",
]

for profile in PROFILES:
    reference_command += ["--profile", profile]

print(" ".join(reference_command))
subprocess.run(reference_command, check=True)

### 4. Compare maps, graphs, and provenance

import pandas as pd
import torch

for day in ("day1", "day2"):
    resumed_map = json.loads(
        (RESUME_TEST_ROOT / "mappings" / f"{day}_ip_to_id.json").read_text()
    )
    reference_map = json.loads(
        (REFERENCE_ROOT / "mappings" / f"{day}_ip_to_id.json").read_text()
    )
    assert resumed_map == reference_map, f"Mapping mismatch: {day}"

resumed_graphs = sorted(
    path.relative_to(RESUME_TEST_ROOT)
    for path in RESUME_TEST_ROOT.rglob("graph_*.pt")
)
reference_graphs = sorted(
    path.relative_to(REFERENCE_ROOT)
    for path in REFERENCE_ROOT.rglob("graph_*.pt")
)

assert resumed_graphs == reference_graphs

tensor_fields = (
    "edge_index",
    "edge_attr",
    "y",
    "global_node_ids",
)

metadata_fields = (
    "num_nodes",
    "timestamp",
    "window_start",
    "window_end",
    "feature_profile",
    "schema_hash",
)

for relative_path in resumed_graphs:
    resumed = torch.load(
        RESUME_TEST_ROOT / relative_path,
        weights_only=False,
    )
    reference = torch.load(
        REFERENCE_ROOT / relative_path,
        weights_only=False,
    )

    for field in tensor_fields:
        torch.testing.assert_close(
            getattr(resumed, field),
            getattr(reference, field),
            rtol=0,
            atol=0,
        )

    for field in metadata_fields:
        assert getattr(resumed, field) == getattr(reference, field)

resumed_provenance = sorted(
    path.relative_to(RESUME_TEST_ROOT)
    for path in (RESUME_TEST_ROOT / "provenance").rglob("graph_*.csv")
)
reference_provenance = sorted(
    path.relative_to(REFERENCE_ROOT)
    for path in (REFERENCE_ROOT / "provenance").rglob("graph_*.csv")
)

assert resumed_provenance == reference_provenance

for relative_path in resumed_provenance:
    resumed = pd.read_csv(RESUME_TEST_ROOT / relative_path)
    reference = pd.read_csv(REFERENCE_ROOT / relative_path)
    pd.testing.assert_frame_equal(resumed, reference)

print("Resume equivalence: passed")
print("Graphs compared:", len(resumed_graphs))
print("Provenance files compared:", len(resumed_provenance))


## Full build

Run this cell only after reviewing the smoke-build output. Use a new versioned output directory instead of overwriting a reviewed graph collection. If Colab disconnects, rerun the same command with `--resume`.

In [36]:
full_command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(FULL_ROOT),
    '--chunksize', '250000',
]

for profile in PROFILES:
    full_command += ['--profile', profile]

subprocess.run(full_command, check=True)

# To resume an interrupted full build, append '--resume' to full_command and run it again.

CompletedProcess(args=['python', '/content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py', '--input-csv', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv', '--corrected-manifest', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json', '--output-root', '/content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1', '--chunksize', '250000', '--profile', 'nfv3_extended', '--profile', 'portable_core'], returncode=0)

In [38]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(REPO_ROOT / "code/python"))
from utils.graph_construction import prepare_chunk

configuration = json.loads(
    (FULL_ROOT / "build_configuration.json").read_text()
)
graph_audit = json.loads(
    (FULL_ROOT / "graph_audit.json").read_text()
)

cutoffs = configuration["day1_cutoffs"]
train_end = int(cutoffs["train_end_ms"])
val_end = int(cutoffs["val_end_ms"])

columns = [
    "source_file",
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "FLOW_DURATION_MILLISECONDS",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "Attack",
    "binary_target",
    "correction_rule",
    "label_corrected_detail",
]

summary_parts = []
transition_parts = []
removed_parts = []
positive_range_parts = []
hourly_parts = []

for chunk in pd.read_csv(
    CORRECTED_CSV,
    usecols=columns,
    chunksize=250_000,
    low_memory=False,
):
    chunk = chunk.loc[
        chunk["source_file"].eq("cicids2018v3_wed2802.csv")
    ]
    if chunk.empty:
        continue

    prepared, _ = prepare_chunk(chunk)
    if prepared.empty:
        continue

    decision_time = prepared["decision_time_ms"].to_numpy(dtype=np.int64)

    split = np.select(
        [decision_time < train_end, decision_time < val_end],
        ["train", "val"],
        default="test1",
    )

    # Exactly replicates the target of the historical builder.
    historical_target = (
        prepared["Attack"]
        .fillna("Benign")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("infilteration")
        .astype(np.int8)
        .to_numpy()
    )

    corrected_target = pd.to_numeric(
        prepared["binary_target"], errors="raise"
    ).to_numpy(dtype=np.int8)

    work = pd.DataFrame({
        "split": split,
        "decision_time_ms": decision_time,
        "historical_target": historical_target,
        "corrected_target": corrected_target,
        "correction_rule": prepared["correction_rule"].astype(str).to_numpy(),
        "corrected_detail": (
            prepared["label_corrected_detail"].astype(str).to_numpy()
        ),
    })

    work["transition"] = (
        work["historical_target"].astype(str)
        + "->"
        + work["corrected_target"].astype(str)
    )

    summary_parts.append(
        work.groupby("split").agg(
            rows=("corrected_target", "size"),
            historical_positive=("historical_target", "sum"),
            corrected_positive=("corrected_target", "sum"),
        )
    )

    transition_parts.append(
        work.groupby(["split", "transition"]).size().rename("rows")
    )

    removed = work.loc[
        work["historical_target"].eq(1)
        & work["corrected_target"].eq(0)
    ]
    if not removed.empty:
        removed_parts.append(
            removed.groupby(["split", "correction_rule"])
            .size()
            .rename("rows")
        )

    positives = work.loc[work["corrected_target"].eq(1)]
    if not positives.empty:
        positive_range_parts.append(
            positives.groupby(["split", "corrected_detail"]).agg(
                positive_rows=("corrected_target", "size"),
                first_decision_ms=("decision_time_ms", "min"),
                last_decision_ms=("decision_time_ms", "max"),
            )
        )

    work["hour_utc"] = (
        pd.to_datetime(work["decision_time_ms"], unit="ms", utc=True)
        .dt.floor("h")
        .astype(str)
    )
    hourly_parts.append(
        work.groupby("hour_utc").agg(
            rows=("corrected_target", "size"),
            historical_positive=("historical_target", "sum"),
            corrected_positive=("corrected_target", "sum"),
        )
    )

split_order = ["train", "val", "test1"]

summary = (
    pd.concat(summary_parts)
    .groupby(level=0)
    .sum()
    .reindex(split_order)
)

transitions = (
    pd.concat(transition_parts)
    .groupby(level=[0, 1])
    .sum()
    .unstack(fill_value=0)
    .reindex(split_order)
)

removed_by_rule = (
    pd.concat(removed_parts)
    .groupby(level=[0, 1])
    .sum()
    .unstack(fill_value=0)
    .reindex(split_order)
)

positive_ranges = (
    pd.concat(positive_range_parts)
    .groupby(level=[0, 1])
    .agg({
        "positive_rows": "sum",
        "first_decision_ms": "min",
        "last_decision_ms": "max",
    })
    .reset_index()
)

positive_ranges["first_utc"] = pd.to_datetime(
    positive_ranges["first_decision_ms"], unit="ms", utc=True
)
positive_ranges["last_utc"] = pd.to_datetime(
    positive_ranges["last_decision_ms"], unit="ms", utc=True
)

hourly = (
    pd.concat(hourly_parts)
    .groupby(level=0)
    .sum()
    .sort_index()
)
hourly_with_positives = hourly.loc[
    hourly["historical_positive"].gt(0)
    | hourly["corrected_positive"].gt(0)
]

# It must exactly match the final graphs.
expected = graph_audit["profiles"]["nfv3_extended"]["splits"]
for split_name in split_order:
    assert int(summary.loc[split_name, "rows"]) == expected[split_name]["edges"]
    assert (
        int(summary.loc[split_name, "corrected_positive"])
        == expected[split_name]["positive_edges"]
    )

print("Cutoffs UTC:")
print("Train ends:", pd.to_datetime(train_end, unit="ms", utc=True))
print("Val ends / Test1 begins:", pd.to_datetime(val_end, unit="ms", utc=True))

print("\nHistorical summary vs. corrected:")
display(summary)

print("\nLabel transitions by split:")
display(transitions)

print("\nHistorical positives eliminated, BY rule.:")
display(removed_by_rule)

print("\nRanges of corrected positive values:")
display(
    positive_ranges[
        [
            "split",
            "corrected_detail",
            "positive_rows",
            "first_utc",
            "last_utc",
        ]
    ]
)

print("\nHourly timeline with some historical or corrected positives:")
display(hourly_with_positives)

diagnostic_path = Path("/content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1/day1_split_label_diagnostic.json")
diagnostic = {
    "train_end_ms": train_end,
    "val_end_ms": val_end,
    "split_summary": summary.reset_index().to_dict(orient="records"),
    "transitions": transitions.reset_index().to_dict(orient="records"),
    "removed_historical_positives_by_rule": (
        removed_by_rule.reset_index().to_dict(orient="records")
    ),
    "corrected_positive_ranges": (
        positive_ranges.assign(
            first_utc=positive_ranges["first_utc"].astype(str),
            last_utc=positive_ranges["last_utc"].astype(str),
        ).to_dict(orient="records")
    ),
    "hourly_positive_timeline": (
        hourly_with_positives.reset_index().to_dict(orient="records")
    ),
}
diagnostic_path.write_text(
    json.dumps(diagnostic, indent=2) + "\n",
    encoding="utf-8",
)

print("\nDiagnostic JSON:", diagnostic_path)


Cutoffs UTC:
Train ends: 2018-02-28 16:53:00+00:00
Val ends / Test1 begins: 2018-02-28 20:27:30+00:00

Historical summary vs. corrected:


,rows,historical_positive,corrected_positive
split,,,
train,1043036,49578,36700
val,789732,31360,27326
test1,189239,3389,0



Label transitions by split:


transition,0->0,0->1,1->0,1->1
split,,,,
train,966075,27383,40261,9317
val,737663,20709,24743,6617
test1,185850,0,3389,0



Historical positives eliminated, BY rule.:


...


Ranges of corrected positive values:


,split,corrected_detail,positive_rows,first_utc,last_utc
0,train,Infiltration - Communication Victim Attacker,6,2018-02-28 14:51:30+00:00,2018-02-28 16:04:30+00:00
1,train,Infiltration - Dropbox Download,9,2018-02-28 14:33:30+00:00,2018-02-28 14:35:00+00:00
2,train,Infiltration - NMAP Portscan,36685,2018-02-28 14:48:00+00:00,2018-02-28 16:05:30+00:00
3,val,Infiltration - Communication Victim Attacker,3,2018-02-28 18:35:30+00:00,2018-02-28 18:38:30+00:00
4,val,Infiltration - NMAP Portscan,27323,2018-02-28 17:46:00+00:00,2018-02-28 18:38:30+00:00



Hourly timeline with some historical or corrected positives:


,rows,historical_positive,corrected_positive
hour_utc,,,
2018-02-28 12:00:00+00:00,175254,2697,0
2018-02-28 13:00:00+00:00,211909,4255,0
2018-02-28 14:00:00+00:00,222423,10208,10828
2018-02-28 15:00:00+00:00,267261,28880,23336
2018-02-28 16:00:00+00:00,183298,3745,2536
2018-02-28 17:00:00+00:00,217801,9588,7122
2018-02-28 18:00:00+00:00,243877,16874,20204
2018-02-28 19:00:00+00:00,210100,3143,0
2018-02-28 20:00:00+00:00,207430,3785,0



Diagnostic JSON: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_v1/day1_split_label_diagnostic.json


In [39]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(REPO_ROOT / "code/python"))
from utils.graph_construction import prepare_chunk


# Candidate division
TRAIN_END_MS = 1519832430000  # 2018-02-28 15:40:30 UTC
VAL_END_MS = 1519839780000    # 2018-02-28 17:43:00 UTC

SPLIT_ORDER = ["train", "val", "test1"]

columns = [
    "source_file",
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "FLOW_DURATION_MILLISECONDS",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "Attack",
    "binary_target",
    "label_corrected_detail",
]

summary_parts = []
transition_parts = []
range_parts = []
hourly_parts = []

for chunk in pd.read_csv(
    CORRECTED_CSV,
    usecols=columns,
    chunksize=250_000,
    low_memory=False,
):
    # Day 1 only
    chunk = chunk.loc[
        chunk["source_file"].eq("cicids2018v3_wed2802.csv")
    ]
    if chunk.empty:
        continue

    # Apply the builder's exclusions and timings exactly.
    prepared, _ = prepare_chunk(chunk)
    if prepared.empty:
        continue

    decision_time = prepared["decision_time_ms"].to_numpy(dtype=np.int64)

    split = np.select(
        [
            decision_time < TRAIN_END_MS,
            decision_time < VAL_END_MS,
        ],
        [
            "train",
            "val",
        ],
        default="test1",
    )

    # Target used by the historical builder.
    historical_target = (
        prepared["Attack"]
        .fillna("Benign")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("infilteration")
        .astype(np.int8)
        .to_numpy()
    )

    corrected_target = pd.to_numeric(
        prepared["binary_target"],
        errors="raise",
    ).to_numpy(dtype=np.int8)

    work = pd.DataFrame({
        "split": split,
        "decision_time_ms": decision_time,
        "historical_target": historical_target,
        "corrected_target": corrected_target,
        "corrected_detail": (
            prepared["label_corrected_detail"]
            .astype(str)
            .to_numpy()
        ),
    })

    work["transition"] = (
        work["historical_target"].astype(str)
        + "->"
        + work["corrected_target"].astype(str)
    )

    summary_parts.append(
        work.groupby("split").agg(
            rows=("corrected_target", "size"),
            historical_positive=("historical_target", "sum"),
            corrected_positive=("corrected_target", "sum"),
        )
    )

    transition_parts.append(
        work.groupby(["split", "transition"])
        .size()
        .rename("rows")
    )

    positives = work.loc[work["corrected_target"].eq(1)]
    if not positives.empty:
        range_parts.append(
            positives.groupby(
                ["split", "corrected_detail"]
            ).agg(
                positive_rows=("corrected_target", "size"),
                first_decision_ms=("decision_time_ms", "min"),
                last_decision_ms=("decision_time_ms", "max"),
            )
        )

    work["hour_utc"] = (
        pd.to_datetime(
            work["decision_time_ms"],
            unit="ms",
            utc=True,
        )
        .dt.floor("h")
        .astype(str)
    )

    hourly_parts.append(
        work.groupby(["split", "hour_utc"]).agg(
            rows=("corrected_target", "size"),
            historical_positive=("historical_target", "sum"),
            corrected_positive=("corrected_target", "sum"),
        )
    )


summary = (
    pd.concat(summary_parts)
    .groupby(level=0)
    .sum()
    .reindex(SPLIT_ORDER)
)

summary["corrected_negative"] = (
    summary["rows"] - summary["corrected_positive"]
)
summary["corrected_positive_pct"] = (
    100 * summary["corrected_positive"] / summary["rows"]
)

transitions = (
    pd.concat(transition_parts)
    .groupby(level=[0, 1])
    .sum()
    .unstack(fill_value=0)
    .reindex(SPLIT_ORDER)
)

positive_ranges = (
    pd.concat(range_parts)
    .groupby(level=[0, 1])
    .agg({
        "positive_rows": "sum",
        "first_decision_ms": "min",
        "last_decision_ms": "max",
    })
    .reset_index()
)

positive_ranges["first_utc"] = pd.to_datetime(
    positive_ranges["first_decision_ms"],
    unit="ms",
    utc=True,
)
positive_ranges["last_utc"] = pd.to_datetime(
    positive_ranges["last_decision_ms"],
    unit="ms",
    utc=True,
)

hourly = (
    pd.concat(hourly_parts)
    .groupby(level=[0, 1])
    .sum()
    .sort_index()
)

hourly_with_positives = hourly.loc[
    hourly["historical_positive"].gt(0)
    | hourly["corrected_positive"].gt(0)
]

# Day 1 Conservation
assert int(summary["rows"].sum()) == 2_022_007
assert int(summary["corrected_positive"].sum()) == 64_026

print("Candidate time cuts:")
print(
    "Train ends:",
    pd.to_datetime(TRAIN_END_MS, unit="ms", utc=True),
)
print(
    "Val ends / Test1 begins:",
    pd.to_datetime(VAL_END_MS, unit="ms", utc=True),
)

print("\nSummary:")
display(summary)

print("\nHistorical transitions → corrected:")
display(transitions)

print("\nPositives corrected for subtype and interval:")
display(
    positive_ranges[
        [
            "split",
            "corrected_detail",
            "positive_rows",
            "first_utc",
            "last_utc",
        ]
    ]
)

print("\nTimeline with positives:")
display(hourly_with_positives)


Candidate time cuts:
Train ends: 2018-02-28 15:40:30+00:00
Val ends / Test1 begins: 2018-02-28 17:43:00+00:00

Summary:


,rows,historical_positive,corrected_positive,corrected_negative,corrected_positive_pct
split,,,,,
train,793679,38055,27096,766583,3.413975
val,409310,14414,9604,399706,2.346388
test1,819018,31858,27326,791692,3.336435



Historical transitions → corrected:


transition,0->0,0->1,1->0,1->1
split,,,,
train,735099,20525,31484,6571
val,388038,6858,11668,2746
test1,766451,20709,25241,6617



Positives corrected for subtype and interval:


,split,corrected_detail,positive_rows,first_utc,last_utc
0,test1,Infiltration - Communication Victim Attacker,3,2018-02-28 18:35:30+00:00,2018-02-28 18:38:30+00:00
1,test1,Infiltration - NMAP Portscan,27323,2018-02-28 17:46:00+00:00,2018-02-28 18:38:30+00:00
2,train,Infiltration - Communication Victim Attacker,3,2018-02-28 14:51:30+00:00,2018-02-28 15:09:30+00:00
3,train,Infiltration - Dropbox Download,9,2018-02-28 14:33:30+00:00,2018-02-28 14:35:00+00:00
4,train,Infiltration - NMAP Portscan,27084,2018-02-28 14:48:00+00:00,2018-02-28 15:30:30+00:00
5,val,Infiltration - Communication Victim Attacker,3,2018-02-28 16:00:30+00:00,2018-02-28 16:04:30+00:00
6,val,Infiltration - NMAP Portscan,9601,2018-02-28 15:57:30+00:00,2018-02-28 16:05:30+00:00



Timeline with positives:


rows  historical_positive  \
split hour_utc                                                 
test1 2018-02-28 17:00:00+00:00   76175                 6904   
      2018-02-28 18:00:00+00:00  243877                16874   
      2018-02-28 19:00:00+00:00  210100                 3143   
      2018-02-28 20:00:00+00:00  207430                 3785   
      2018-02-28 21:00:00+00:00   77501                 1152   
train 2018-02-28 12:00:00+00:00  175254                 2697   
      2018-02-28 13:00:00+00:00  211909                 4255   
      2018-02-28 14:00:00+00:00  222423                10208   
      2018-02-28 15:00:00+00:00  182875                20895   
val   2018-02-28 15:00:00+00:00   84386                 7985   
      2018-02-28 16:00:00+00:00  183298                 3745   
      2018-02-28 17:00:00+00:00  141626                 2684   

                                 corrected_positive  
split hour_utc                                       
test1 2018-02-28 17:00:00+00:00                7122  
      2018-02-28 18:00:00+00:00               20204  
      2018-02-28 19:00:00+00:00                   0  
      2018-02-28 20:00:00+00:00                   0  
      2018-02-28 21:00:00+00:00                   0  
train 2018-02-28 12:00:00+00:00                   0  
      2018-02-28 13:00:00+00:00                   0  
      2018-02-28 14:00:00+00:00               10828  
      2018-02-28 15:00:00+00:00               16268  
val   2018-02-28 15:00:00+00:00                7068  
      2018-02-28 16:00:00+00:00                2536  
      2018-02-28 17:00:00+00:00                   0